In [1]:
!pip install -q transformers datasets torchaudio librosa scikit-learn accelerate torchcodec

In [2]:
import os
import random
import torch
import torchaudio
import numpy as np


from torch.utils.data import Dataset
from transformers import (
Wav2Vec2Processor,
Wav2Vec2ForSequenceClassification,
TrainingArguments,
DataCollatorWithPadding,
Trainer
)


from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [3]:
# -------------------------
# Configuración general
# -------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATASET_PATH = "/content/dataset"
SAMPLE_RATE = 16000
NUM_LABELS = 3
SEED = 42


random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
# -------------------------
# Mapeo de emociones (3 clases)
# -------------------------
emotion2id = {
    "happiness": 0,      # Positivo
    "neutral": 1,      # Neutral
    "sadness": 2,     # Negativo
    "anger": 2,
    "disgust": 2,
    "fear": 2
}

id2emotion = {
    0: "positivo",
    1: "neutral",
    2: "negativo"
}

In [5]:
# -------------------------
# Dataset
# -------------------------
class EmotionDataset(Dataset):
    def __init__(self, files, data_dir, processor):
        self.files = files
        self.data_dir = data_dir
        self.processor = processor

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        filename = self.files[idx]
        path = os.path.join(self.data_dir, filename)

        # Etiqueta desde el nombre
        name = os.path.splitext(filename)[0]   # elimina extensión
        emotion = name.split("_")[0].lower()
        label = emotion2id[emotion]

        # Cargar audio
        speech, sr = torchaudio.load(path)

        if sr != SAMPLE_RATE:
            speech = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(speech)

        speech = speech.mean(dim=0)  # mono

        inputs = self.processor(
            speech.numpy(),
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
            padding=False
        )

        return {
            "input_values": inputs.input_values.squeeze(),
            "labels": torch.tensor(label)
        }


In [6]:
# -------------------------
# Modelo y procesador
# -------------------------
MODEL_NAME = "jonatasgrosman/wav2vec2-large-xlsr-53-spanish"

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)

#data_collator = data_collator

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
).to(DEVICE)

# Congelar extractor (clave para corpus pequeño)
for param in model.wav2vec2.feature_extractor.parameters():
    param.requires_grad = False

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/262 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: jonatasgrosman/wav2vec2-large-xlsr-53-spanish
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
projector.weight  | MISSING    | 
projector.bias    | MISSING    | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
# -------------------------
# División del corpus
# -------------------------
all_files = [f for f in os.listdir(DATASET_PATH) if f.lower().endswith(".wav")]
random.shuffle(all_files)

print("Archivos detectados:", len(all_files))
print(all_files[:5])


train_size = int(0.8 * len(all_files))
train_files = all_files[:train_size]
test_files = all_files[train_size:]

print(f"Train: {len(train_files)} | Test: {len(test_files)}")

train_dataset = EmotionDataset(train_files, DATASET_PATH, processor)
test_dataset = EmotionDataset(test_files, DATASET_PATH, processor)


Archivos detectados: 862
['Fear_M_B_ataque.wav', 'Disgust_M_A_basta_ya.wav', 'Neutral_C_A_lento.wav', 'Anger_M_B_ira.wav', 'Fear_M_A_ayer.wav']
Train: 689 | Test: 173


In [8]:
# -------------------------
# Métricas
# -------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1-score": f1
    }

In [9]:
# -------------------------
# Argumentos de entrenamiento
# -------------------------
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=8,
    warmup_steps=10,
    logging_dir="./logs",
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="f1-score",
    fp16=False
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [10]:
# -------------------------
# Trainer
# -------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=processor,
    compute_metrics=compute_metrics
)
# -------------------------
# Entrenamiento y evaluación
# -------------------------
trainer.train()

results = trainer.evaluate()
print("\nResultados finales:")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

# ============================
# BASELINE ALEATORIA
# ============================

print("\n===== BASELINE ALEATORIA =====")

# Obtener etiquetas reales del test set
true_labels = []

for i in range(len(test_dataset)):
    true_labels.append(test_dataset[i]["labels"].item())

true_labels = np.array(true_labels)

# Fijar semilla para reproducibilidad
np.random.seed(SEED)

# Generar predicciones aleatorias (3 clases)
random_preds = np.random.randint(0, NUM_LABELS, size=len(true_labels))

# Calcular métricas
precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels,
    random_preds,
    average="weighted",
    zero_division=0
)

acc = accuracy_score(true_labels, random_preds)

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")



Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1-score
1,1.225465,0.878095,0.763006,0.798668,0.763006,0.732774
2,0.711218,1.144097,0.635838,0.800705,0.635838,0.658789
3,0.010157,1.261515,0.716763,0.814654,0.716763,0.735496
4,1.516215,0.701466,0.872832,0.883281,0.872832,0.868102
5,1.442627,0.526477,0.890173,0.899468,0.890173,0.890747
6,0.003453,0.525819,0.895954,0.901224,0.895954,0.895969


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1-score
1,1.225465,0.878095,0.763006,0.798668,0.763006,0.732774
2,0.711218,1.144097,0.635838,0.800705,0.635838,0.658789
3,0.010157,1.261515,0.716763,0.814654,0.716763,0.735496
4,1.516215,0.701466,0.872832,0.883281,0.872832,0.868102
5,1.442627,0.526477,0.890173,0.899468,0.890173,0.890747
6,0.003453,0.525819,0.895954,0.901224,0.895954,0.895969
7,1.620831,0.575559,0.907514,0.914427,0.907514,0.907120
8,1.545765,0.576589,0.907514,0.911679,0.907514,0.906620


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Resultados finales:
eval_loss: 0.5756
eval_accuracy: 0.9075
eval_precision: 0.9144
eval_recall: 0.9075
eval_f1-score: 0.9071
eval_runtime: 4.4900
eval_samples_per_second: 38.5300
eval_steps_per_second: 38.5300
epoch: 8.0000

===== BASELINE ALEATORIA =====
Accuracy : 0.3121
Precision: 0.3997
Recall   : 0.3121
F1-score : 0.3352
